# API Support Debug Lab — Colab walkthrough

A reproducible developer-support debugging lab written in Rust. This notebook walks the crate end-to-end on a fresh Colab VM in roughly **two minutes** after the Rust toolchain installs.

**What you'll see.** Eight diagnostic rules. Fourteen bundled positive fixtures plus eleven negatives. Three real-API webhook envelopes (Stripe v1, Slack v0, GitHub HMAC). A Brier-calibrated confidence model with a regression canary. Ninety-plus tests across the library, CLI, schema, calibration, snapshot, property, oracle, and latency suites, all green.

**What this is not.** A web service. A live API. A learned classifier. The lab is local, offline, rule-based, and honest about every claim it makes.


## Code map and run guide

The repository has two layers: a small diagnostic library and a thin CLI wrapper.

- `src/cases.rs` defines the `Case` data model, loads `case.json`, discovers sibling `server.log` / `secret.txt` files, validates the on-disk layout through tests, and falls back to embedded fixtures when the binary is installed without a repository checkout.
- `src/rules.rs` is the rule engine. It registers eight deterministic rules, parses text or JSON-lines logs, recomputes HMAC-SHA256 signatures, derives retry elapsed time from RFC3339 timestamps, compares idempotency body hashes, and sorts diagnoses by confidence with deterministic tie-breaking.
- `src/evidence.rs` keeps each evidence item tied to a source pointer such as `request.headers.authorization` or a log line number.
- `src/report.rs` renders the same diagnosis as human text or JSON and builds byte-stable curl reproductions.
- `src/main.rs` is the CLI: `list-cases`, `diagnose`, `explain`, `replay`, `report`, and `corpus`.
- `fixtures/cases/` contains positive fixtures. `_negatives/` contains lookalike cases that must remain unclassified. `_calibration/` adds labelled edge cases for confidence calibration.
- `tests/` covers rule behavior, schema validation, CLI behavior, snapshots, property tests, HMAC oracle vectors, calibration metrics, and latency budget.

Installed from crates.io, the bundled fixtures are embedded in the binary:

```bash
cargo install api-debug-lab
api-debug-lab list-cases
api-debug-lab diagnose auth_missing
api-debug-lab diagnose webhook_signature_invalid_stale --format json | jq
```

From a source checkout, use Cargo directly and point `corpus` at the fixture tree:

```bash
cargo run -- list-cases
cargo run -- diagnose auth_missing
cargo run -- corpus fixtures/cases | tail -25
cargo test
```

To diagnose your own captures, create the same directory shape (`case.json`, optional `server.log`, optional `secret.txt`) and pass either a fixture root or a direct path:

```bash
api-debug-lab --fixtures /path/to/fixtures diagnose my_case
api-debug-lab diagnose /path/to/my_case/case.json
api-debug-lab corpus /path/to/case-directory
```


In [ ]:
# Install a minimal Rust toolchain. ~90 s on Colab the first time.
# `--profile minimal` skips docs and the rust-src component we don't need here.
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!rustc --version && cargo --version

In [ ]:
# Clone the repo. Requires the repo to be reachable at this URL.
!git clone --depth 1 https://github.com/infinityabundance/api-support-debug-lab.git
%cd api-support-debug-lab

In [ ]:
# Release build so the demo runs fast (~30 s wall-clock on Colab).
# `tail -5` keeps the output cell tight; the build itself is uneventful.
!cargo build --release 2>&1 | tail -5

In [ ]:
# Fourteen bundled positive fixtures, one per directory under fixtures/cases/.
# Negatives and the _calibration set are hidden from this listing by convention.
!./target/release/api-debug-lab list-cases


In [ ]:
# The money shot: the human report a support engineer would paste into a ticket.
# Notice the EVIDENCE block (three structural signals, not just '401'), the
# REPRODUCTION block (a deterministic curl command), the NEXT STEPS block,
# and the ESCALATION NOTE block that names the divergence space.
!./target/release/api-debug-lab diagnose auth_missing

In [ ]:
# Two rules fire on this case: webhook_signature_mismatch (HMAC differs) AND
# webhook_timestamp_stale (drift outside tolerance). The orchestrator ranks
# them by confidence — HMAC mismatch is dispositive evidence, timestamp
# drift has benign causes — and surfaces the loser as ALSO CONSIDERED.
!./target/release/api-debug-lab diagnose webhook_signature_invalid_stale

In [ ]:
# Slack-style envelope: `X-Slack-Signature: v0=<hex>` over the signing input
# `"v0:{timestamp}:{body}"`. The rule recomputes the HMAC against the
# bundled secret and reports the mismatch. The two other supported envelopes
# (Stripe v1, GitHub HMAC) are exercised by their own fixtures.
!./target/release/api-debug-lab diagnose webhook_slack_v0

In [ ]:
# Sweep every case.json under fixtures/cases/ (positives, negatives, and
# the _calibration edge cases). Exit code is non-zero if any case is
# unclassified — useful as a regression check when adding rules.
!./target/release/api-debug-lab corpus fixtures/cases | tail -25

In [ ]:
# Full test suite: per-rule unit tests, schema validation, CLI integration
# via assert_cmd, snapshot tests via insta, property-based tests via proptest,
# calibration (aggregate Brier, per-rule Brier, ECE), oracle differential
# tests against externally-computed HMAC references, and a per-rule latency
# budget. ~5 s on Colab.
!cargo test --tests 2>&1 | grep 'test result'

In [ ]:
# The calibration test enforces five empirical properties over the 36-case
# labelled corpus: 100% primary-classification accuracy, aggregate Brier
# ≤ 0.05, per-rule Brier ≤ 0.08, ECE ≤ 0.05, and zero confidence on
# unclassified cases. The rubric is documented in docs/confidence_model.md.
!cargo test --test calibration 2>&1 | tail -15

## Deeper rigour cells

These are normal code cells, so Colab **Run all** executes them. They add several minutes because they install extra Cargo tools and run heavier empirical checks. Measured baseline numbers live in `docs/mutation_report.md`, `docs/coverage.md`, and `docs/benchmarks.md`; the confidence rubric is in `docs/confidence_model.md`; architecture decisions live under `docs/adr/`.


In [ ]:
# Mutation testing: validates that tests catch rule-level behavioral changes.
# The shared Colab VM is too noisy for the microsecond latency-budget test
# during cargo-mutants' baseline run, so this cell skips only that perf test.
# Baseline report: 91% kill rate over 169 viable mutants in src/rules.rs.
!cargo install --locked cargo-mutants
!cargo mutants --in-place --file src/rules.rs --no-shuffle --timeout-multiplier=2 -- --tests -- --skip per_rule_median_within_budget


In [ ]:
# Coverage summary over the default test suite.
# Colab is too noisy for the microsecond latency-budget test, so this
# skips only that perf test while preserving functional coverage.
# Baseline report: ~92% regions and ~93% lines.
!cargo install --locked cargo-llvm-cov
!rustup component add llvm-tools-preview
!cargo llvm-cov --summary-only -- --skip per_rule_median_within_budget


In [ ]:
# Microsecond per-case benchmark smoke.
!cargo bench --bench diagnose -- --quick 2>&1 | grep 'time:'


In [ ]:
# Calibration regression canary: injects a deliberately miscalibrated rule
# and asserts the production Brier check would catch it.
!cargo test --features calibration_canary --test calibration_regression
